# Shape and breaks along an ordered axis

The `ordered_axis` detector states what a numeric response does
along an ordered axis: how its typical value moves with the axis
(shape), and where along the axis its level, trend, or regression
slope changes (breaks). The axis does not have to be time. It is,
in precedence order: a column named by the `axis` hint; a date or
datetime column; an integer-year column; a strictly increasing
integer sequence column; a categorical column whose level order
is declared with the `ordinal` hint. Shape statements also run on
any other numeric column as the axis — dose, age, income, mileage —
with no declaration at all.


## The statements

Five statement families come out of one battery:

- `axis_shape` — the shape of the response along the axis, from
  the medians of 20 equal-frequency bins: *monotone* when the
  Spearman correlation and an isotonic fit both clear their bars;
  *monotone with reversal blocks* when an adjacent bin-median move
  against the trend exceeds a quarter of the response's
  interquartile range (the blocks are named — a bare correlation
  would hide exactly the part that matters); *non-monotone with a
  located reversal* when a block exceeds half the interquartile
  range; *saturating* when a continuous two-segment fit on the bin
  medians bends to a post-knee slope at most 0.35 of the pre-knee
  slope and the bend is stable under 100 row bootstraps (at least
  70% classify the pair as saturating and the knee's bootstrap
  interquartile range is at most 15% of the axis range). When the
  fit bends but the knee is not stable, the weaker wording *rises,
  then flattens* is used and no knee is located. Flatness is a
  ratio claim; the plateau's absolute rise is in the evidence.
- `level_shift` and `trend_change` — the breaks detector's forms,
  stated by this battery on axes and slices that detector does
  not scan: supported numeric axes, and single categorical levels
  (a per-slice statement, below).
- `slope_break` — a located change in the slope of the response's
  regression on another numeric column, or on the axis itself:
  a single-break segmented fit whose breakpoint range holds every
  split within 3.84 residual variances of the best, kept only when
  the two segment slopes' confidence intervals do not overlap.
- `regime_contrast` — a fixed-window contrast: the mean over a
  window against the mean over the window before it. The windows
  are fixed by the series length and anchored at break locations
  proposed by the scans or at calendar boundaries; they are never
  searched. When the series reverts, the two contrasts merge into
  one episode statement with its end.
- `spike_contrast` — a window of at most 6 axis points whose mean
  jumps against a season-matched baseline (the same window one
  seasonal period earlier) and reverts afterwards. Without
  reversion the event is a level shift and the level scan owns
  it; the word *spike* is not used.

A per-slice break is a marginal statement: one categorical
column, one level, that level's own series against the pooled
series' break profile. It is stated only when the pooled series
does not show the same event at the same location (then the event
is global), and it is suppressed when the `ordered_partitions`
detector splits on the same dimension with a leaf boundary at
the same location, because the partition leaf already states it
jointly. Panel entity keys are not slices; per-entity breaks
belong to the `series` detector.


## Calibration and selection

Every statistic is calibrated against a shuffled-axis null: the
axis values are permuted, the statistic is recomputed, and the
observed value is read against the null's maximum (with a Gumbel
tail fit when the observed value exceeds every null maximum).
The Benjamini–Hochberg procedure then runs over the whole
evaluated family — all statistics, responses, axes, and slices —
and only statements with q at most 0.05 are kept. The family size
is in each finding's evidence.

Nearby firings describe one event, so they are merged before
anything is stated: firings whose locations are within a merging
radius (the larger of 6 axis points and 3% of the axis length),
or whose windows overlap, form one cluster and produce one
statement. The primary form is chosen by a fixed priority —
spike, then regime contrast, then slope or trend break, then
level scan — and the other firings are named in the statement's
`corroboration` evidence. A cluster that an earlier pass (breaks,
changeforest, ordered partitions) already stated for the same
response is not restated. At most 3 events are stated per
response, axis, and slice.

On a time axis spanning fewer than 2 full seasonal cycles, the
scans stay silent: on a short window a level scan mostly detects
the season itself. Only the contrasts may fire there, because
their baselines are season-matched by construction.

`report.verify` marks these statements `not_checkable`. Their
locations come from calibrated searches over the data, not from
counts that can be recounted exactly; the evidence records the
calibration so the statement can be re-derived, not recounted.


## Example: a saturating response

A response that rises with dose and levels off: 500 units
approached exponentially, with noise. The planted curve's bend
region is around dose 0.25 to 0.4.


In [1]:
import math
import random

import polars as pl

import onceover

rng = random.Random(273277)
rows = []
for _ in range(10_000):
    dose = rng.random()
    y = 500.0 * (1.0 - math.exp(-dose / 0.25)) + rng.gauss(0.0, 60.0)
    rows.append((dose, y))
df = pl.DataFrame(rows, schema=["dose", "y"], orient="row")

report = onceover.profile(df, detectors=["ordered_axis"])
for f in report.select(type="axis_shape"):
    print(f["claim"])


y rises with dose and saturates above dose ≈ 0.327589: slope 1027.927191 per unit before the knee, 181.96411 after (0.17702 of the earlier slope); the plateau still rises 0.700094 IQR across its span
dose rises monotonically with y (Spearman rho 0.797481); no reversal block exceeds a quarter of its interquartile range


The statement says *saturates* and locates the knee, because
every bar holds on this fixture: the correlation is strong, the
two-segment fit improves on one line, the post-knee slope is a
small fraction of the pre-knee slope, and the bootstrap finds
the same bend at nearly the same place. A near-linear pair fails
the slope-ratio bar no matter how stable its knee is, and stays
silent — the ratio bar, not the stability bar, is what separates
saturation from a straight line.


## Example: two events on one series

A monthly series with a permanent level shift at month 300 and a
three-month spike at month 700. The battery states two events,
one per cluster, with the spike as a spike (it reverts) and the
shift in a level form. The shift also draws a shape statement:
a permanent step is the limiting case of a saturating rise, and
shape statements are a separate claim family from event
statements, so both are kept.


In [2]:
import datetime as dt
import random

import polars as pl

import onceover

rng = random.Random(277)
months = []
yy, mm = 2000, 1
for _ in range(900):
    months.append(dt.date(yy, mm, 1))
    mm += 1
    if mm == 13:
        yy, mm = yy + 1, 1
y = []
for i in range(900):
    v = rng.gauss(0.0, 1.0)
    if i >= 300:
        v += 2.0
    if 700 <= i < 703:
        v += 6.0
    y.append(v)
df = pl.DataFrame({"month": months, "y": y})

report = onceover.profile(df, detectors=["ordered_axis"])
for f in report.findings:
    print(f["type"], "-", f["claim"])


axis_shape - y rises with month, then flattens above month ≈ 2043-02-01; the knee location is not stable under resampling, so no saturation is claimed
regime_contrast - y averaged 2.188522 over the 24 points from 2025-01-01, against -0.084299 over the 24 points before it — a regime contrast of +2.272822
spike_contrast - y spikes over 2058-05-01 to 2058-07-01: 8.611306 against a season-matched baseline of 2.085052 (+6.526254), reverting to 2.94742 in the following window


## Read the finding

The evidence of a break statement carries the located breakpoint
(or the window, for a contrast), the segment or window means and
slopes, the shuffled-axis calibration, the Benjamini–Hochberg
q-value and the family size it was adjusted in, and the
`corroboration` list naming any other statistics that fired at
the same event and were merged into the statement. A per-slice
statement adds `slice_column` and `slice_value`. A shape
statement carries the bin-median reversal blocks as a payload,
and a saturation statement the knee, the two slopes and their
ratio, and the bootstrap shares.


In [3]:
f = report.select(type="spike_contrast")[0]
import json

print(json.dumps(f["evidence"][0]["metrics"], indent=1)[:1500])


{
 "axis": "month",
 "baseline_kind": "season_matched",
 "baseline_mean": 2.0850524428091357,
 "bh_q": 0.009975948786212774,
 "corroboration": [],
 "family_size": 10,
 "jump": 6.526253574599972,
 "n_axis_points": 900,
 "n_rows": 900,
 "n_used": 900,
 "response": "y",
 "reversion": 0.8623674613001229,
 "statistic": "spike_contrast",
 "window_end": "2058-07-01",
 "window_end_index": 703,
 "window_mean": 8.611306017409108,
 "window_start": "2058-05-01",
 "window_start_index": 700
}


## Limits

- One break per scan in this version; a series with two level
  shifts is stated through the breaks detector's recursive scan
  on time axes, and multi-break segmentation of a regression is
  not attempted.
- Windows for contrasts are fixed by the series length (4% of
  its points, between 6 and 24); a regime much shorter or longer
  than the window is sized approximately, not exactly.
- Break scans on a non-time numeric axis need at least 48
  distinct axis values with at least 5 rows at the median value;
  below that support the axis carries shape statements only.
- The battery aggregates rows sharing an axis value by their
  mean before scanning; statements are about the typical response
  at each axis value, not about individual rows.
